<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/18-causal-machine-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Causal Machine Learning**

Predictive machine learning learns stable associations that help forecast an outcome. **Causal machine learning** asks a different question: how would an outcome change if a treatment, policy, price, message, or system component were deliberately changed? Flexible prediction remains useful, but it is embedded inside a causal design rather than treated as evidence of causation by itself.

A causal analysis should begin with four objects:

1. a well-defined **treatment or intervention** $T$;
2. an **outcome** $Y$ measured over a specified time horizon;
3. a **target population** to which the conclusion applies;
4. a **causal estimand**, such as an average or conditional treatment effect.

Only after these are fixed should an analyst choose an estimator. “Use a causal forest” is not a causal question, just as “use a neural network” is not a prediction problem definition.

A rigorous workflow separates five layers that are often collapsed in applied work:

| Layer | Main question | Typical output |
|---|---|---|
| Specification | What intervention, population, outcome horizon, and effect are intended? | Target trial and estimand |
| Causal model or design | Which variables cause treatment and outcome, or which assignment mechanism creates comparison? | DAG, randomized design, or quasi-experimental design |
| Identification | Can the causal estimand be written using the observed-data distribution? | Adjustment formula, instrumental-variable estimand, RD contrast, or proof of non-identification |
| Estimation | How should the identified quantity be approximated from finite data? | Regression, weighting, matching, orthogonal score, or forest estimate |
| Inference and stress testing | How uncertain is the estimate, and how sensitive is it to violations? | Standard error, confidence interval, diagnostics, placebo checks, and sensitivity analysis |

The distinction is not cosmetic. Identification is a logical result under assumptions, whereas estimation is a statistical approximation. A low-variance estimator of the wrong identified quantity remains precisely wrong.

### **Prediction, Association, and Causation**

The same variables can support three distinct questions:

| Question type | Mathematical object | Example | What validates it? |
|---|---|---|---|
| Prediction | $P(Y\mid X)$ or $\mathbb E[Y\mid X]$ | Who is likely to churn? | Out-of-sample predictive performance under the deployment distribution |
| Association | Contrast across observed groups | How does churn differ between contacted and uncontacted users? | Accurate description of the observed data |
| Intervention | $P(Y\mid do(T=t))$ or $\mathbb E[Y(t)]$ | What would churn be if everyone received contact strategy $t$? | Study design plus identification assumptions |
| Counterfactual | $Y_i(t)$ relative to another unrealized treatment | Would this user have stayed without the contact they received? | A structural or potential-outcome model with stronger assumptions |

<div class="diagram-scroll">

![Prediction, intervention, and counterfactual questions form different levels of causal reasoning.](assets/causal-question-ladder.svg){fig-alt="Prediction estimates likely outcomes, intervention asks what changes when a treatment is set, and counterfactual reasoning compares realized and unrealized outcomes for a unit."}

</div>

Association and causal effect coincide only under an appropriate design or adjustment argument. If high-risk users are more likely to receive an intervention, the treated group can have worse outcomes even when treatment helps. Conversely, a beneficial association can be produced entirely by favorable selection.

<details>
<summary><strong>Python: show how confounding separates association from treatment effect</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
n = 20_000

# Engagement affects both campaign assignment and future spending.
engagement = rng.normal(size=n)
propensity = 1 / (1 + np.exp(-(-0.2 + 1.4 * engagement)))
treatment = rng.binomial(1, propensity)

true_effect = 2.0
outcome = (
    5.0
    + true_effect * treatment
    + 3.0 * engagement
    + rng.normal(scale=1.0, size=n)
)

# This is only an observed-group contrast.
naive_difference = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()

# Under the simulated no-hidden-confounding assumption, adjusting for engagement
# recovers the treatment coefficient.
adjusted_model = LinearRegression().fit(
    np.column_stack([treatment, engagement]),
    outcome,
)
adjusted_effect = adjusted_model.coef_[0]

print(f"True causal effect:        {true_effect:.3f}")
print(f"Naive treated-control gap: {naive_difference:.3f}")
print(f"Adjusted treatment effect: {adjusted_effect:.3f}")
```

</details>

The adjusted coefficient is causal here only because the simulation makes engagement the complete set of common causes and uses a correctly specified additive outcome model. A higher cross-validated $R^2$ would not prove those assumptions.


### **Structural Causal Models**

A **structural causal model (SCM)** represents each endogenous variable as a deterministic function of its direct causes and an exogenous disturbance:

$$
X_j=f_j(\operatorname{pa}_j,U_j).
$$

The collection

$$
\mathcal M=(U,V,F,P_U)
$$

contains exogenous variables $U$, endogenous variables $V$, structural mechanisms $F$, and a distribution over exogenous disturbances $P_U$. In an acyclic SCM, evaluating the equations in topological order generates the observational distribution.

The key causal assumption is **modularity**: each structural equation describes an autonomous mechanism that can, at least conceptually, be changed without rewriting every other mechanism. This is what gives an arrow a causal meaning beyond statistical dependence.

#### **Causal DAGs and Structural Equations**

A directed acyclic graph (DAG) summarizes which variables appear as direct causes in the structural equations. For example,

$$
C=f_C(U_C),\qquad
T=f_T(C,U_T),\qquad
Y=f_Y(T,C,U_Y)
$$

induces $C\rightarrow T$, $C\rightarrow Y$, and $T\rightarrow Y$. The graph omits the exact functional form but exposes pathways relevant to identification.

An intervention $do(T=t)$ creates a modified model by replacing the natural treatment equation with

$$
T:=t.
$$

All arrows entering $T$ are removed, while the equations for $C$ and $Y$ remain unchanged.

<div class="diagram-scroll">

![A surgical intervention replaces the treatment equation while preserving the other mechanisms.](assets/scm-surgical-intervention.svg){fig-alt="The observational SCM is transformed by replacing the natural treatment mechanism with a fixed treatment value, producing an interventional world."}

</div>

<details>
<summary><strong>Python: compare an observational slope with a surgical intervention</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(7)
n = 50_000

# U is a common cause of treatment and outcome.
u = rng.normal(size=n)
treatment_noise = rng.normal(size=n)
outcome_noise = rng.normal(size=n)

treatment = 1.2 * u + treatment_noise
outcome = 3.0 * treatment + 2.0 * u + outcome_noise

observational_slope = LinearRegression().fit(
    treatment.reshape(-1, 1), outcome
).coef_[0]
adjusted_slope = LinearRegression().fit(
    np.column_stack([treatment, u]), outcome
).coef_[0]

# do(T=t) replaces the treatment equation but leaves U and the outcome
# mechanism unchanged. Using common noise makes the contrast easy to audit.
y_do_0 = 3.0 * 0.0 + 2.0 * u + outcome_noise
y_do_1 = 3.0 * 1.0 + 2.0 * u + outcome_noise
interventional_effect = np.mean(y_do_1 - y_do_0)

print(f"Observational slope:       {observational_slope:.3f}")
print(f"Adjusted slope:            {adjusted_slope:.3f}")
print(f"E[Y|do(T=1)]-E[Y|do(T=0)]: {interventional_effect:.3f}")
```

</details>

A DAG is not learned merely by drawing arrows after seeing correlations. Arrow directions encode temporal, scientific, engineering, or design knowledge. Two analysts can fit the same observed distribution while making different causal claims because their models disagree about unobserved mechanisms.

#### **Confounders, Mediators, and Colliders**

A variable's role is determined by the path under study, not by its column name:

- A **confounder** is a common cause on a backdoor path, such as $T\leftarrow C\rightarrow Y$. Adjusting for an adequate set of pre-treatment confounders can block noncausal association.
- A **mediator** lies on a causal pathway, such as $T\rightarrow M\rightarrow Y$. Adjusting for $M$ removes mediated effect and therefore does not estimate the total effect.
- A **collider** has two arrows entering it, such as $T\rightarrow S\leftarrow Y$. The path is naturally closed, but conditioning on $S$ can open an artificial association.
- A **proxy** may imperfectly measure a confounder without fully controlling it.
- An **instrument** affects treatment but, under strong exclusion and independence conditions, affects the outcome only through treatment.

<div class="diagram-scroll">

![Confounders, mediators, and colliders require different adjustment decisions.](assets/dag-variable-roles.svg){fig-alt="A confounder causes treatment and outcome, a mediator transmits part of the treatment effect, and a collider is caused by both treatment and outcome."}

</div>

These roles can be made precise with **d-separation**. A path is blocked by an adjustment set $Z$ if at least one of the following holds:

1. the path contains a non-collider that belongs to $Z$;
2. the path contains a collider for which neither the collider nor any descendant of it belongs to $Z$.

If neither condition holds, the path is open, or **d-connected**, given $Z$. Conditioning on a non-collider blocks a path; conditioning on a collider or its descendant can open one. This reversal is why variable selection based only on association, feature importance, or predictive gain is not a valid adjustment strategy.

For the total effect of $T$ on $Y$, a **proper backdoor path** begins with an arrow entering $T$. A sufficient adjustment set blocks every such path while excluding descendants of $T$. There may be several valid sets. A **minimal sufficient set** is one for which removing any variable would reopen a backdoor path; it is often preferable because unnecessary adjustment can increase variance, amplify measurement error, or make overlap worse.

| Variable relative to the $T\rightarrow Y$ question | Adjust for the total effect? | Reason |
|---|---|---|
| Pre-treatment common cause of $T$ and $Y$ | Usually yes | Blocks a noncausal backdoor path |
| Cause of $Y$ unrelated to $T$ | Optional | Can improve precision without changing identification |
| Strong cause of $T$ unrelated to $Y$ | Usually unnecessary | May increase weight variance and amplify residual hidden bias |
| Mediator caused by $T$ | No | Removes part of the total causal pathway |
| Collider or descendant of a collider | No | Can create selection-induced association |
| Descendant of treatment measured before the outcome | No for the total effect | It is already post-treatment even if its timestamp precedes $Y$ |

“Control for every available feature” is therefore unsafe. Features recorded after treatment can be mediators, colliders, consequences of selection, or leakage from the outcome. Adjustment sets should be justified from the causal question and graph before model fitting.


### **Interventions and the Do-Operator**

The conditional distribution

$$
P(Y\mid T=t)
$$

describes units whose treatment naturally equals $t$. The interventional distribution

$$
P(Y\mid do(T=t))
$$

describes a modified system in which treatment is externally set to $t$. The two are generally different because conditioning does not remove the causes of treatment.

#### **Observational and Interventional Distributions**

If a DAG factorizes as

$$
P(v_1,\ldots,v_p)=\prod_{j=1}^{p}P(v_j\mid \operatorname{pa}_j),
$$

then intervening on $T$ produces a **truncated factorization**: the factor for the natural treatment mechanism is removed, $T$ is fixed, and every other mechanism is retained.

<div class="diagram-scroll">

![Conditioning selects naturally treated units whereas intervention changes treatment assignment.](assets/conditioning-versus-intervention.svg){fig-alt="Conditioning keeps the treatment's natural causes active, while an intervention cuts incoming treatment arrows and forces its value."}

</div>

Identification asks whether the interventional target can be rewritten entirely in terms of the observed-data distribution under stated assumptions. Estimation then approximates that identified expression from finite data. This ordering matters:

> An unidentified effect cannot be rescued by a more accurate predictive model.

#### **Backdoor and Frontdoor Adjustment**

If a pre-treatment set $C$ blocks every backdoor path from $T$ to $Y$ and contains no descendant of $T$, then

$$
P(Y\mid do(T=t))
=
\sum_c P(Y\mid T=t,C=c)P(C=c).
$$

For a continuous or high-dimensional $C$, the sum becomes an integral estimated through outcome regression, weighting, matching, stratification, or a doubly robust method.

<div class="diagram-scroll">

![Backdoor adjustment uses observed common causes, whereas frontdoor identification uses a valid mediator.](assets/backdoor-frontdoor-identification.svg){fig-alt="The backdoor graph adjusts for a common cause of treatment and outcome; the frontdoor graph uses a fully mediating variable despite hidden treatment-outcome confounding."}

</div>

<details>
<summary><strong>Python: estimate a backdoor-adjusted ATE by outcome standardization</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(12)
n = 12_000
x = rng.normal(size=(n, 3))

# Treatment selection depends nonlinearly on observed pre-treatment covariates.
logit = 0.2 + 0.9 * x[:, 0] - 0.7 * x[:, 1] + 0.4 * x[:, 0] * x[:, 1]
propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, propensity)

true_cate = 1.5 + 0.8 * np.tanh(x[:, 0])
baseline = 2.0 + x[:, 0] ** 2 - 0.8 * x[:, 1] + 0.5 * x[:, 2]
outcome = baseline + true_cate * treatment + rng.normal(scale=1.0, size=n)

# Learn E[Y | T, X], then standardize both treatment states over the same X.
outcome_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=20,
    random_state=12,
    n_jobs=-1,
).fit(np.column_stack([treatment, x]), outcome)

mu_1 = outcome_model.predict(np.column_stack([np.ones(n), x]))
mu_0 = outcome_model.predict(np.column_stack([np.zeros(n), x]))

naive = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()
g_formula_ate = np.mean(mu_1 - mu_0)

print(f"True sample ATE:       {true_cate.mean():.3f}")
print(f"Naive observed gap:    {naive:.3f}")
print(f"Standardized RF ATE:   {g_formula_ate:.3f}")
```

</details>

The **frontdoor criterion** can identify an effect despite unobserved $T$-$Y$ confounding when a mediator $M$ satisfies strict conditions: $T$ causally affects $M$ without unblocked confounding, $M$ intercepts every directed path from $T$ to $Y$, and all backdoor paths from $M$ to $Y$ are blocked by $T$. Then

$$
P(y\mid do(t))
=
\sum_m P(m\mid t)
\sum_{t'}P(y\mid m,t')P(t').
$$

<details>
<summary><strong>Python: recover a frontdoor effect in a simulated system</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

rng = np.random.default_rng(21)
n = 60_000

# U confounds treatment and outcome but does not directly affect the mediator.
u = rng.normal(size=n)
p_t = 1 / (1 + np.exp(-(1.2 * u)))
treatment = rng.binomial(1, p_t)

p_m = 1 / (1 + np.exp(-(-0.6 + 1.8 * treatment)))
mediator = rng.binomial(1, p_m)
outcome = 2.5 * mediator + 1.5 * u + rng.normal(size=n)

mediator_model = LogisticRegression().fit(treatment.reshape(-1, 1), mediator)
outcome_model = LinearRegression().fit(
    np.column_stack([mediator, treatment]), outcome
)

treatment_probability = treatment.mean()

def frontdoor_mean(t_value):
    # First average over M under the intervention T=t.
    p_m1 = mediator_model.predict_proba([[t_value]])[0, 1]
    total = 0.0
    for m_value, p_m_given_t in [(0, 1 - p_m1), (1, p_m1)]:
        # Then adjust the M->Y relation over the natural distribution of T.
        y_m_t0 = outcome_model.predict([[m_value, 0]])[0]
        y_m_t1 = outcome_model.predict([[m_value, 1]])[0]
        adjusted_y_m = (
            (1 - treatment_probability) * y_m_t0
            + treatment_probability * y_m_t1
        )
        total += p_m_given_t * adjusted_y_m
    return total

frontdoor_effect = frontdoor_mean(1) - frontdoor_mean(0)
true_effect = 2.5 * (
    1 / (1 + np.exp(-1.2)) - 1 / (1 + np.exp(0.6))
)
naive = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()

print(f"True total effect:  {true_effect:.3f}")
print(f"Naive group gap:    {naive:.3f}")
print(f"Frontdoor estimate: {frontdoor_effect:.3f}")
```

</details>

Frontdoor adjustment is not a generic fix for hidden confounding. Its assumptions are often harder to defend than ordinary backdoor adjustment, especially the absence of a direct treatment-outcome path and the absence of hidden treatment-mediator confounding.


### **Potential Outcomes**

The potential-outcomes framework defines, for each unit $i$,

$$
Y_i(1)
\quad\text{and}\quad
Y_i(0),
$$

the outcomes that would occur under treatment and control. For binary treatment, the observed outcome satisfies **consistency**:

$$
Y_i
=
T_iY_i(1)+(1-T_i)Y_i(0).
$$

This notation defines a causal effect without requiring a particular estimator. It also makes the missing-data structure explicit.

#### **Counterfactuals and the Fundamental Problem of Causal Inference**

For one unit, the individual treatment effect is

$$
\tau_i=Y_i(1)-Y_i(0).
$$

Only one potential outcome is observed, so $\tau_i$ is never directly observed. This is the **fundamental problem of causal inference**. Repeated measurements usually do not solve it because time, carryover, learning, and changing context create different units or treatment regimes.

<div class="diagram-scroll">

![Treatment assignment reveals one potential outcome and leaves the other counterfactual.](assets/potential-outcomes-missingness.svg){fig-alt="Each unit has a treated and untreated potential outcome, but treatment assignment reveals only one, so the individual treatment effect is missing."}

</div>

The common SUTVA shorthand combines:

- **consistency**: the observed outcome under the received treatment equals the corresponding potential outcome;
- **no hidden versions of treatment**: treatment $t$ represents a sufficiently well-defined intervention;
- **no interference**: one unit's outcome does not depend on other units' assignments.

Network effects, shared resources, auctions, epidemics, and recommendation systems frequently violate no-interference assumptions. The treatment must then include relevant exposure mappings or operate at a cluster or network level.

#### **Identification Assumptions**

Potential outcomes define the estimand, but they do not by themselves identify it. For an observational binary treatment, the standard adjustment argument combines:

1. **consistency**: if $T=t$, then $Y=Y(t)$;
2. **conditional exchangeability**: $Y(t)\perp T\mid X$ for $t\in\{0,1\}$;
3. **positivity**: every covariate profile in the target population has a nonzero probability of each treatment;
4. **well-defined sampling and measurement**: $X$, $T$, and $Y$ represent the intended population, treatment versions, and outcome horizon.

Under these conditions, the mean potential outcome is identified by the g-formula:

$$
\begin{aligned}
\mathbb E[Y(t)]
&=\mathbb E_X\!\left[\mathbb E[Y(t)\mid X]\right] \\
&=\mathbb E_X\!\left[\mathbb E[Y(t)\mid T=t,X]\right] \\
&=\mathbb E_X\!\left[\mathbb E[Y\mid T=t,X]\right].
\end{aligned}
$$

The first line applies iterated expectation, the second uses exchangeability, and the third uses consistency. Positivity is what makes the last conditional expectation empirically learnable for both treatment states over the target covariate distribution.

This derivation also reveals distinct failure modes. If exchangeability fails, the observed conditional mean is not the missing counterfactual mean. If positivity fails, identification may hold algebraically for a larger hypothetical population but the data contain no support for estimating it without extrapolation. If treatment has hidden versions, $Y(t)$ is not a single coherent potential outcome.

#### **ATE, ATT, and Heterogeneous Treatment Effects**

Population summaries include

$$
\operatorname{ATE}
=
\mathbb E[Y(1)-Y(0)],
$$

$$
\operatorname{ATT}
=
\mathbb E[Y(1)-Y(0)\mid T=1],
$$

and the conditional average treatment effect

$$
\tau(x)
=
\mathbb E[Y(1)-Y(0)\mid X=x].
$$

ATE, ATT, and ATC differ whenever treatment effects are heterogeneous and treatment selection changes the composition of the groups.

<div class="diagram-scroll">

![ATE, ATT, ATC, and CATE average treatment effects over different populations.](assets/causal-estimands-populations.svg){fig-alt="ATE targets the full population, ATT targets treated units, ATC targets untreated units, and CATE targets units with a specified covariate value."}

</div>

<details>
<summary><strong>Python: show why ATE, ATT, and the observed group difference diverge</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(30)
n = 100_000
x = rng.normal(size=n)

y0 = 2.0 + x + rng.normal(scale=0.8, size=n)
individual_effect = 1.0 + 1.2 * x
y1 = y0 + individual_effect

# Units with larger effects are more likely to receive treatment.
propensity = 1 / (1 + np.exp(-(-0.3 + 1.2 * x)))
treatment = rng.binomial(1, propensity)
observed_y = treatment * y1 + (1 - treatment) * y0

ate = individual_effect.mean()
att = individual_effect[treatment == 1].mean()
atc = individual_effect[treatment == 0].mean()
observed_gap = (
    observed_y[treatment == 1].mean()
    - observed_y[treatment == 0].mean()
)

print(f"ATE:                    {ate:.3f}")
print(f"ATT:                    {att:.3f}")
print(f"ATC:                    {atc:.3f}")
print(f"Observed treated gap:   {observed_gap:.3f}")
```

</details>

An estimated CATE is a conditional mean, not proof that an individual's effect is known. Heterogeneity analysis also creates multiple-testing, regularization, overlap, and policy-evaluation problems. A subgroup discovered and evaluated on the same data can look responsive by construction.


### **Randomized and Observational Studies**

Randomization and observational adjustment are not interchangeable data-cleaning choices. They are different identification strategies.

<div class="diagram-scroll">

![Randomized experiments, observational studies, and target-trial emulation place different burdens on assumptions.](assets/randomized-observational-target-trial.svg){fig-alt="Randomized assignment identifies effects by design, observational assignment requires assumptions, and target-trial emulation makes eligibility, time zero, treatment, and outcome explicit."}

</div>

#### **Randomization and Identification**

In an ideal randomized experiment,

$$
T\perp (Y(1),Y(0),X),
$$

so the difference in sample means identifies the ATE:

$$
\widehat{\tau}_{\mathrm{DM}}
=
\overline Y_{T=1}-\overline Y_{T=0}.
$$

Covariate adjustment can improve precision, but randomization is the source of identification. Practical complications still matter:

- noncompliance separates assignment effects from treatment-received effects;
- attrition can reintroduce selection;
- treatment spillovers violate no interference;
- repeated peeking or adaptive stopping changes inference;
- cluster assignment requires cluster-aware standard errors;
- an intention-to-treat estimand differs from a per-protocol effect.

The **target-trial** framework forces an observational analysis to state the experiment it is trying to emulate:

| Protocol component | Question that must be answered |
|---|---|
| Eligibility | Who enters the target population, and when are eligibility variables measured? |
| Time zero | At what instant are eligibility, treatment assignment, and follow-up aligned? |
| Treatment strategies | What exactly counts as treatment and control, including dose, timing, and grace periods? |
| Assignment | What randomized mechanism is imagined, and which measured variables are needed to emulate it? |
| Follow-up | When does risk begin and end; how are censoring and competing events handled? |
| Outcome | Which endpoint, measurement procedure, and horizon define success? |
| Estimand | ITT, per-protocol, ATE, ATT, survival contrast, or another policy-relevant effect? |
| Analysis | How will confounding, censoring, missingness, repeated treatment, and uncertainty be handled? |

Misaligning time zero is especially dangerous. For example, classifying patients as “treated” only after they survive long enough to receive treatment gives the treated group an **immortal time** during which the outcome could not yet have occurred. No flexible learner repairs that design error.

<details>
<summary><strong>Python: compare randomized and selectively assigned treatment across repeated studies</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(44)
replications = 500
n = 1_000
true_ate = 2.0

randomized_estimates = []
observational_estimates = []

for _ in range(replications):
    x = rng.normal(size=n)
    treatment_effect = true_ate + 0.5 * x
    baseline = 1.5 * x + rng.normal(size=n)

    randomized_t = rng.binomial(1, 0.5, size=n)
    randomized_y = baseline + randomized_t * treatment_effect
    randomized_estimates.append(
        randomized_y[randomized_t == 1].mean()
        - randomized_y[randomized_t == 0].mean()
    )

    propensity = 1 / (1 + np.exp(-1.4 * x))
    observational_t = rng.binomial(1, propensity)
    observational_y = baseline + observational_t * treatment_effect
    observational_estimates.append(
        observational_y[observational_t == 1].mean()
        - observational_y[observational_t == 0].mean()
    )

print(
    "Randomized mean estimate / bias:",
    f"{np.mean(randomized_estimates):.3f}",
    f"/ {np.mean(randomized_estimates) - true_ate:.3f}",
)
print(
    "Observational mean gap / bias:",
    f"{np.mean(observational_estimates):.3f}",
    f"/ {np.mean(observational_estimates) - true_ate:.3f}",
)
print(
    "Randomized sampling SD:",
    f"{np.std(randomized_estimates, ddof=1):.3f}",
)
```

</details>

For observational data, a common identification condition is conditional exchangeability:

$$
(Y(1),Y(0))\perp T\mid X,
$$

combined with consistency and positivity:

$$
0<P(T=1\mid X=x)<1
$$

throughout the target population. Exchangeability cannot be verified from observed data alone; it is a claim that $X$ captures the relevant common causes.

Quasi-experimental designs replace exchangeability with different assumptions:

| Design | Source of identifying variation | Key threat |
|---|---|---|
| Instrumental variables | A valid instrument changes treatment | Exclusion, independence, monotonicity, weak instruments |
| Regression discontinuity | Assignment changes at a threshold | Manipulation and continuity near the cutoff |
| Difference-in-differences | Treated and comparison trends provide a counterfactual | Parallel trends and treatment timing |
| Synthetic control | Weighted donor units approximate the untreated trajectory | Donor support and time-varying confounding |

These designs target specific populations and effects. They should not be presented as interchangeable estimators selected by validation score.

#### **Matching, Stratification, and Propensity Scores**

The **propensity score**

$$
e(X)=P(T=1\mid X)
$$

is a balancing score: under conditional exchangeability, units with the same propensity have the same distribution of measured baseline covariates across treatment groups. It is not the probability that treatment is causally beneficial.

<div class="diagram-scroll">

![Treated and untreated propensity distributions require common support.](assets/propensity-overlap.svg){fig-alt="The treated and untreated propensity-score distributions overlap only in a common-support region, outside which causal comparisons require extrapolation."}

</div>

Matching constructs explicit comparisons; stratification compares outcomes within propensity or covariate blocks. Both require diagnostics:

- standardized mean differences before and after adjustment;
- overlap and unmatched units;
- the number of times controls are reused;
- sensitivity to calipers, distance metrics, and replacement;
- a variance estimate that respects the design.

For a continuous covariate $X_j$, the unweighted standardized mean difference is

$$
\operatorname{SMD}_j
=
\frac{\overline X_{1j}-\overline X_{0j}}
{\sqrt{(s_{1j}^{2}+s_{0j}^{2})/2}}.
$$

Weighted or matched means and variances replace the raw group summaries after adjustment. SMD is a scale-free balance diagnostic, not a hypothesis test; with a large sample, a trivial imbalance can be statistically significant, while with a small sample, an important imbalance can have a large $p$-value.

The matching design determines the estimand. Matching every treated unit to one or more controls naturally targets an ATT-like population. Dropping treated units outside common support changes that population. Matching with replacement can improve match quality but lets a few controls represent many treated units, so uncertainty must account for reuse. A caliper prevents visibly poor matches, but a narrow caliper can discard much of the original population.

A propensity model should be selected for balance and overlap, not treatment-classification accuracy. Perfect treatment prediction is evidence of poor positivity, not a modeling victory.

<details>
<summary><strong>Python: match on propensity and inspect covariate balance</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(52)
n = 6_000
x = rng.normal(size=(n, 3))
true_effect = 2.0

logit = -0.2 + 1.1 * x[:, 0] - 0.9 * x[:, 1] + 0.5 * x[:, 2]
propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, propensity)
outcome = (
    true_effect * treatment
    + 1.8 * x[:, 0]
    - 1.2 * x[:, 1]
    + 0.5 * x[:, 2] ** 2
    + rng.normal(size=n)
)

estimated_propensity = LogisticRegression(max_iter=2_000).fit(
    x, treatment
).predict_proba(x)[:, 1]

treated_idx = np.flatnonzero(treatment == 1)
control_idx = np.flatnonzero(treatment == 0)
matcher = NearestNeighbors(n_neighbors=1).fit(
    estimated_propensity[control_idx].reshape(-1, 1)
)
_, nearest = matcher.kneighbors(
    estimated_propensity[treated_idx].reshape(-1, 1)
)
matched_control_idx = control_idx[nearest[:, 0]]

def standardized_mean_difference(a, b):
    pooled_sd = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_sd

before = [
    standardized_mean_difference(x[treated_idx, j], x[control_idx, j])
    for j in range(x.shape[1])
]
after = [
    standardized_mean_difference(x[treated_idx, j], x[matched_control_idx, j])
    for j in range(x.shape[1])
]
matched_att = np.mean(outcome[treated_idx] - outcome[matched_control_idx])

balance = pd.DataFrame(
    {"covariate": ["X1", "X2", "X3"], "SMD before": before, "SMD after": after}
)
print(balance.round(3).to_string(index=False))
print(f"True ATT:      {true_effect:.3f}")
print(f"Matched ATT:   {matched_att:.3f}")
```

</details>

#### **Inverse Probability Weighting**

For a binary treatment, inverse probability weighting (IPW) uses

$$
w_i^{\mathrm{ATE}}
=
\frac{T_i}{e(X_i)}
+
\frac{1-T_i}{1-e(X_i)}.
$$

The weighted treated and control groups approximate a pseudo-population in which measured baseline covariates are independent of treatment.

The Horvitz-Thompson ATE estimator is

$$
\widehat\tau_{\mathrm{HT}}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left[
\frac{T_iY_i}{\widehat e_i}
-
\frac{(1-T_i)Y_i}{1-\widehat e_i}
\right].
$$

The Hájek version normalizes treated and control weights separately. It is not exactly unbiased in finite samples, but it is often less sensitive to random fluctuations in the total weight. Other targets imply other weights:

| Target | Treated weight | Control weight | Interpretation |
|---|---:|---:|---|
| ATE | $1/e(X)$ | $1/[1-e(X)]$ | Reconstruct the full target population under both treatments |
| ATT | $1$ | $e(X)/[1-e(X)]$ | Reweight controls to resemble treated units |
| ATC | $[1-e(X)]/e(X)$ | $1$ | Reweight treated units to resemble controls |
| Overlap population | $1-e(X)$ | $e(X)$ | Emphasize units with genuine treatment ambiguity |

Weight diagnostics should include the maximum and upper quantiles, treatment-specific sums, covariate balance after weighting, and the effective sample size

$$
n_{\mathrm{eff}}
=
\frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}.
$$

A dataset with 50,000 rows can contain the weighted information of only a few hundred balanced observations when a small number of extreme weights dominate.

<div class="diagram-scroll">

![Inverse propensity weights create a balanced pseudo-population.](assets/ipw-pseudopopulation.svg){fig-alt="Observed units are weighted by inverse treatment propensity to create comparable treated and control pseudo-populations."}

</div>

<details>
<summary><strong>Python: estimate an ATE with stabilized means and diagnose weight quality</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(63)
n = 10_000
x = rng.normal(size=(n, 4))
true_effect = 1.75

logit = -0.1 + 1.0 * x[:, 0] - 0.8 * x[:, 1] + 0.5 * x[:, 2]
true_propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, true_propensity)
outcome = (
    true_effect * treatment
    + 2.0 * x[:, 0]
    - 1.0 * x[:, 1]
    + 0.4 * x[:, 2] ** 2
    + rng.normal(size=n)
)

e_hat = LogisticRegression(max_iter=2_000).fit(
    x, treatment
).predict_proba(x)[:, 1]

# Positivity diagnostics are reported before any optional clipping.
raw_weights = treatment / e_hat + (1 - treatment) / (1 - e_hat)
effective_sample_size = raw_weights.sum() ** 2 / np.sum(raw_weights ** 2)

treated_weights = treatment / e_hat
control_weights = (1 - treatment) / (1 - e_hat)
hajek_ate = (
    np.sum(treated_weights * outcome) / np.sum(treated_weights)
    - np.sum(control_weights * outcome) / np.sum(control_weights)
)

print(f"True ATE:              {true_effect:.3f}")
print(f"IPW Hajek ATE:         {hajek_ate:.3f}")
print(f"Propensity range:      [{e_hat.min():.3f}, {e_hat.max():.3f}]")
print(f"Maximum ATE weight:    {raw_weights.max():.1f}")
print(f"Effective sample size: {effective_sample_size:.0f} / {n}")
```

</details>

Extreme weights reveal weak overlap and unstable extrapolation. Trimming or clipping can reduce variance, but it changes the effective target population and introduces bias. Report both the rule and the fraction of observations affected.

#### **Augmented IPW and Doubly Robust Estimation**

Outcome regression estimates $\mu_t(x)=\mathbb E[Y\mid T=t,X=x]$; IPW estimates the treatment mechanism $e(x)$. **Augmented inverse probability weighting (AIPW)** combines both through the score

$$
\widehat\phi_i
=
\widehat\mu_1(X_i)-\widehat\mu_0(X_i)
+
\frac{T_i\{Y_i-\widehat\mu_1(X_i)\}}{\widehat e(X_i)}
-
\frac{(1-T_i)\{Y_i-\widehat\mu_0(X_i)\}}{1-\widehat e(X_i)},
$$

and estimates the ATE with

$$
\widehat\tau_{\mathrm{AIPW}}
=
\frac{1}{n}\sum_{i=1}^{n}\widehat\phi_i.
$$

The first term is the outcome-model contrast. The two residual corrections use observed outcomes to repair systematic errors in that contrast. Under regularity and positivity, the estimator is **doubly robust**: it is consistent if either the outcome models or the propensity model are consistently estimated. This does not mean two poor models somehow guarantee a correct answer.

The score is also Neyman orthogonal, so sufficiently accurate nuisance estimates can be learned with flexible ML while preserving first-order inference for the low-dimensional ATE. In finite samples, nuisance predictions should be made out of fold: each observation's propensity and potential-outcome predictions come from models that did not train on that observation.

<details>
<summary><strong>Python: implement cross-fitted AIPW and inspect double robustness</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(64)
n = 12_000
x = rng.normal(size=(n, 5))

# The propensity belongs to the logistic model, while the outcome surface
# is nonlinear. The treatment effect is heterogeneous but averages to 1.5.
true_logit = -0.2 + 0.8 * x[:, 0] - 0.7 * x[:, 1] + 0.4 * x[:, 2]
true_propensity = 1 / (1 + np.exp(-true_logit))
treatment = rng.binomial(1, true_propensity)
mu_0 = (
    1.0
    + np.sin(x[:, 0])
    + 0.8 * x[:, 1] ** 2
    - 0.5 * x[:, 2] * x[:, 3]
)
true_cate = 1.5 + 0.4 * x[:, 0]
outcome = mu_0 + treatment * true_cate + rng.normal(size=n)

e_correct = np.empty(n)
e_poor = np.empty(n)
mu0_flexible = np.empty(n)
mu1_flexible = np.empty(n)
mu0_poor = np.empty(n)
mu1_poor = np.empty(n)

kfold = KFold(n_splits=4, shuffle=True, random_state=64)
for fold, (train_idx, test_idx) in enumerate(kfold.split(x)):
    x_train, x_test = x[train_idx], x[test_idx]
    t_train, y_train = treatment[train_idx], outcome[train_idx]

    propensity_model = LogisticRegression(max_iter=2_000).fit(
        x_train, t_train
    )
    e_correct[test_idx] = propensity_model.predict_proba(x_test)[:, 1]
    e_poor[test_idx] = DummyClassifier(strategy="prior").fit(
        x_train, t_train
    ).predict_proba(x_test)[:, 1]

    for t_value, flexible_target, poor_target in [
        (0, mu0_flexible, mu0_poor),
        (1, mu1_flexible, mu1_poor),
    ]:
        in_group = t_train == t_value
        flexible_model = HistGradientBoostingRegressor(
            max_iter=180,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=1.0,
            random_state=10 * fold + t_value,
        ).fit(x_train[in_group], y_train[in_group])
        poor_model = DummyRegressor(strategy="mean").fit(
            x_train[in_group], y_train[in_group]
        )
        flexible_target[test_idx] = flexible_model.predict(x_test)
        poor_target[test_idx] = poor_model.predict(x_test)

def aipw_summary(mu1_hat, mu0_hat, e_hat):
    # Clipping here only guards numerical explosions in the demonstration;
    # a real analysis must report overlap and any changed target population.
    e_hat = np.clip(e_hat, 0.02, 0.98)
    plugin = np.mean(mu1_hat - mu0_hat)
    ipw = np.mean(
        treatment * outcome / e_hat
        - (1 - treatment) * outcome / (1 - e_hat)
    )
    score = (
        mu1_hat
        - mu0_hat
        + treatment * (outcome - mu1_hat) / e_hat
        - (1 - treatment) * (outcome - mu0_hat) / (1 - e_hat)
    )
    estimate = score.mean()
    standard_error = score.std(ddof=1) / np.sqrt(n)
    return plugin, ipw, estimate, standard_error

configurations = [
    ("both informative", mu1_flexible, mu0_flexible, e_correct),
    ("outcome informative only", mu1_flexible, mu0_flexible, e_poor),
    ("propensity informative only", mu1_poor, mu0_poor, e_correct),
    ("both poor", mu1_poor, mu0_poor, e_poor),
]

rows = []
for label, mu1_hat, mu0_hat, e_hat in configurations:
    plugin, ipw, aipw, se = aipw_summary(mu1_hat, mu0_hat, e_hat)
    rows.append(
        {
            "nuisance models": label,
            "plugin": plugin,
            "IPW": ipw,
            "AIPW": aipw,
            "AIPW SE": se,
        }
    )

print(f"True sample ATE: {true_cate.mean():.3f}")
print(pd.DataFrame(rows).round(3).to_string(index=False))
```

</details>

The empirical standard deviation of the cross-fitted scores divided by $\sqrt n$ gives an influence-function standard error under independent sampling and regularity conditions. Clustered assignment, repeated observations, survey sampling, or time dependence require a variance estimator that respects that dependence. The resulting interval quantifies sampling uncertainty **conditional on the identification assumptions**; it does not include uncertainty about hidden confounding.


### **Causal Machine Learning Estimators**

Causal machine learning uses flexible prediction to estimate nuisance functions such as

$$
\mu_t(x)=\mathbb E[Y\mid T=t,X=x]
\quad\text{and}\quad
e(x)=P(T=1\mid X=x),
$$

or to model heterogeneous effects directly. These algorithms improve functional flexibility; they do not make hidden-confounding, consistency, positivity, or sampling assumptions disappear.

#### **Uplift and Meta-Learners**

**Uplift modeling** ranks units by incremental response rather than predicted outcome. A user with high purchase probability under both treatment and control is a strong predictor target but may have near-zero uplift.

Meta-learners organize standard supervised learners:

- **S-learner**: fit one model $\hat\mu(x,t)$ and toggle treatment at prediction time;
- **T-learner**: fit separate models $\hat\mu_1(x)$ and $\hat\mu_0(x)$;
- **X-learner**: impute treatment effects in each group, fit effect models, and combine them using propensity information;
- **R-learner**: residualize treatment and outcome, then learn the effect function from an orthogonalized loss.

<div class="diagram-scroll">

![S-, T-, and X-learners organize supervised outcome and effect models differently.](assets/causal-meta-learners.svg){fig-alt="The S-learner uses one outcome model, the T-learner uses separate treatment and control models, and the X-learner imputes and combines treatment effects."}

</div>

<details>
<summary><strong>Python: compare S-, T-, and X-learners on known heterogeneous effects</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(71)
n = 8_000
x = rng.normal(size=(n, 4))
treatment = rng.binomial(1, 0.5, size=n)

mu_0 = 1.0 + x[:, 0] ** 2 - 0.5 * x[:, 1] + np.sin(x[:, 2])
true_cate = 1.0 + 0.8 * x[:, 0] + 0.7 * (x[:, 1] > 0)
outcome = mu_0 + treatment * true_cate + rng.normal(scale=1.0, size=n)

x_train, x_test, t_train, _, y_train, _, _, tau_test = train_test_split(
    x, treatment, outcome, true_cate, test_size=0.35, random_state=71
)

def forest(seed):
    return RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=15,
        random_state=seed,
        n_jobs=-1,
    )

# S-learner
s_model = forest(1).fit(np.column_stack([x_train, t_train]), y_train)
s_tau = (
    s_model.predict(np.column_stack([x_test, np.ones(len(x_test))]))
    - s_model.predict(np.column_stack([x_test, np.zeros(len(x_test))]))
)

# T-learner
model_0 = forest(2).fit(x_train[t_train == 0], y_train[t_train == 0])
model_1 = forest(3).fit(x_train[t_train == 1], y_train[t_train == 1])
t_tau = model_1.predict(x_test) - model_0.predict(x_test)

# X-learner with randomized propensity e(x)=0.5
mu0_for_treated = model_0.predict(x_train[t_train == 1])
mu1_for_control = model_1.predict(x_train[t_train == 0])
d_treated = y_train[t_train == 1] - mu0_for_treated
d_control = mu1_for_control - y_train[t_train == 0]
tau_1 = forest(4).fit(x_train[t_train == 1], d_treated)
tau_0 = forest(5).fit(x_train[t_train == 0], d_control)
x_tau = 0.5 * tau_0.predict(x_test) + 0.5 * tau_1.predict(x_test)

def rmse(estimate):
    return np.sqrt(np.mean((estimate - tau_test) ** 2))

print(f"S-learner CATE RMSE: {rmse(s_tau):.3f}")
print(f"T-learner CATE RMSE: {rmse(t_tau):.3f}")
print(f"X-learner CATE RMSE: {rmse(x_tau):.3f}")
print("One simulation is not a universal algorithm ranking.")
```

</details>

#### **Evaluating Heterogeneous Effects and Policies**

CATE evaluation is difficult because $Y_i(1)-Y_i(0)$ is never observed for an individual. Ordinary RMSE cannot be computed on real outcomes, and a model can predict $Y$ accurately while ranking treatment effects badly. Evaluation should distinguish three questions:

| Evaluation goal | Question | Defensible evidence |
|---|---|---|
| Calibration | Do groups assigned a predicted effect near $a$ actually have an average effect near $a$? | Honest bins compared with experimental or doubly robust effect estimates |
| Ranking | Does the model place more responsive units above less responsive units? | Uplift/Qini curves, RATE or AUTOC summaries, and uncertainty on a holdout sample |
| Decision value | Does a policy using the scores improve expected utility after treatment cost and constraints? | Off-policy value estimation or a prospective randomized policy test |

Synthetic and semi-synthetic benchmarks reveal true CATE and are useful for debugging, but they can reward assumptions that real data do not satisfy. On randomized holdout data with known propensity $e$, the transformed outcome

$$
\Gamma_i
=
\frac{T_iY_i}{e}
-
\frac{(1-T_i)Y_i}{1-e}
$$

satisfies $\mathbb E[\Gamma_i\mid X_i=x]=\tau(x)$. It is noisy at the unit level, so it should be averaged over sufficiently large, pre-specified score groups rather than treated as an individual label. In observational data, a cross-fitted AIPW pseudo-outcome is usually preferable.

For a binary policy $\pi(X)\in\{0,1\}$ and known treatment propensity, its value can be estimated by

$$
\widehat V(\pi)
=
\frac{1}{n}
\sum_{i=1}^{n}
\frac{\mathbb I\{T_i=\pi(X_i)\}Y_i}
{P(T_i\mid X_i)}.
$$

If treatment has a cost, subtract it from the treated potential outcome or include it in the utility before comparing policies. A policy should be learned on one sample and evaluated on an independent sample or through nested cross-fitting.

<details>
<summary><strong>Python: evaluate CATE calibration and policy value on a randomized holdout set</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(75)

def make_randomized_data(n):
    x = rng.normal(size=(n, 4))
    treatment = rng.binomial(1, 0.5, size=n)
    mu0 = 1.0 + x[:, 0] ** 2 - 0.5 * x[:, 1] + np.sin(x[:, 2])
    cate = 0.3 + 1.0 * np.tanh(x[:, 0]) + 0.6 * (x[:, 1] > 0)
    outcome = mu0 + treatment * cate + rng.normal(scale=1.0, size=n)
    return x, treatment, outcome, mu0, cate

x_train, t_train, y_train, _, _ = make_randomized_data(8_000)
x_test, t_test, y_test, mu0_test, cate_test = make_randomized_data(12_000)

def forest(seed):
    return RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=25,
        random_state=seed,
        n_jobs=-1,
    )

# Fit a T-learner only on the training sample.
model_0 = forest(1).fit(x_train[t_train == 0], y_train[t_train == 0])
model_1 = forest(2).fit(x_train[t_train == 1], y_train[t_train == 1])
cate_hat = model_1.predict(x_test) - model_0.predict(x_test)

# With randomized e=0.5, this noisy pseudo-outcome has conditional mean CATE.
transformed_outcome = (
    t_test * y_test / 0.5
    - (1 - t_test) * y_test / 0.5
)

calibration = pd.DataFrame(
    {
        "predicted CATE": cate_hat,
        "experimental effect": transformed_outcome,
        "true CATE (simulation only)": cate_test,
    }
)
calibration["score decile"] = pd.qcut(
    calibration["predicted CATE"], 10, labels=False
)
calibration_table = calibration.groupby("score decile").mean()

# Treat only when estimated benefit exceeds treatment cost.
treatment_cost = 0.40
policy = (cate_hat > treatment_cost).astype(int)
observed_net_outcome = y_test - treatment_cost * t_test

def randomized_policy_value(policy_action):
    # The indicator keeps outcomes from units whose randomized action agrees
    # with the policy; division by 0.5 reconstructs the full population.
    return np.mean(
        (t_test == policy_action) * observed_net_outcome / 0.5
    )

estimated_policy_value = randomized_policy_value(policy)
estimated_treat_all = randomized_policy_value(np.ones(len(policy), dtype=int))
estimated_treat_none = randomized_policy_value(np.zeros(len(policy), dtype=int))
true_policy_value = np.mean(mu0_test + policy * (cate_test - treatment_cost))

print(calibration_table.round(3).to_string())
print(f"\nEstimated learned-policy value: {estimated_policy_value:.3f}")
print(f"True learned-policy value:      {true_policy_value:.3f}")
print(f"Estimated treat-all value:      {estimated_treat_all:.3f}")
print(f"Estimated treat-none value:     {estimated_treat_none:.3f}")
```

</details>

<div class="diagram-scroll">

![An EconML doubly robust CATE interpreter summarizes estimated effect heterogeneity with an intentionally shallow tree.](assets/econml-dr-cate-tree.png){fig-alt="A shallow interpretation tree partitions a population into subgroups with different mean CATE estimates, uncertainty summaries, and sample counts." width="95%"}

</div>

*Example output from the official [EconML repository](https://github.com/py-why/EconML/blob/main/notebooks/images/dr_cate_tree.png), distributed under the repository's [MIT license](https://github.com/py-why/EconML/blob/main/LICENSE). The shallow tree is an interpretation aid for an underlying CATE model, not proof that the split variables cause the outcome.*

Calibration, ranking, and value can disagree. A model can rank effects well while shrinking their magnitude, which may preserve a top-$k$ policy but misstate cost thresholds. Conversely, a globally calibrated model can miss useful heterogeneity. Report uncertainty, overlap, subgroup sizes, and stability across honest splits rather than selecting an attractive subgroup after seeing its estimated effect.

#### **Double Machine Learning**

Consider the partially linear model

$$
Y=\theta_0 T+g_0(X)+\zeta,
\qquad
T=m_0(X)+V,
$$

where $\mathbb E[\zeta\mid T,X]=0$ and $\mathbb E[V\mid X]=0$. Define

$$
\ell_0(X)=\mathbb E[Y\mid X]
=
\theta_0m_0(X)+g_0(X).
$$

A plug-in regression can inherit regularization bias from flexible estimates of $\ell_0$ and $m_0$. Double/debiased machine learning (DML) uses the orthogonal score

$$
\psi(W;\theta,\eta)
=
\left(T-m(X)\right)
\left[
Y-\ell(X)-\theta\left(T-m(X)\right)
\right].
$$

The target solves $\mathbb E[\psi(W;\theta_0,\eta_0)]=0$. Neyman orthogonality means that the expected score is locally insensitive to nuisance perturbations:

$$
\left.
\frac{\partial}{\partial r}
\mathbb E\!\left[
\psi\!\left(W;\theta_0,\eta_0+r(\eta-\eta_0)\right)
\right]
\right|_{r=0}
=0.
$$

This removes first-order regularization bias; second-order products of nuisance errors remain. **Cross-fitting** prevents the same observation from being used both to fit a high-capacity nuisance model and to evaluate its residual:

1. split observations into $K$ folds;
2. fit $\widehat\ell^{(-k)}$ and $\widehat m^{(-k)}$ without fold $k$;
3. predict fold $k$ and construct $\widetilde Y_i=Y_i-\widehat\ell^{(-k)}(X_i)$ and $\widetilde T_i=T_i-\widehat m^{(-k)}(X_i)$;
4. pool all out-of-fold residuals and solve $\sum_i\widetilde T_i(\widetilde Y_i-\theta\widetilde T_i)=0$.

<div class="diagram-scroll">

![Double machine learning constructs out-of-fold treatment and outcome residuals before estimating the causal parameter.](assets/double-ml-cross-fitting.svg){fig-alt="Outcome and treatment nuisance models are trained on separate folds, their out-of-fold residuals are formed, and an orthogonal score estimates the treatment effect."}

</div>

<details>
<summary><strong>Python: implement cross-fitted residual-on-residual DML</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(82)
n = 10_000
p = 8
x = rng.normal(size=(n, p))
theta_true = 2.25

m_x = 0.8 * np.sin(x[:, 0]) + 0.5 * x[:, 1] * x[:, 2] - 0.3 * x[:, 3] ** 2
g_x = 2.0 * m_x + 1.2 * np.cos(x[:, 0]) + x[:, 1] ** 2 + 0.7 * x[:, 4] * x[:, 5]
treatment = m_x + rng.normal(size=n)
outcome = theta_true * treatment + g_x + rng.normal(scale=2.0, size=n)

outcome_residual = np.empty(n)
treatment_residual = np.empty(n)
kfold = KFold(n_splits=5, shuffle=True, random_state=82)

for train_idx, test_idx in kfold.split(x):
    model_y = HistGradientBoostingRegressor(
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        random_state=1,
    ).fit(x[train_idx], outcome[train_idx])
    model_t = HistGradientBoostingRegressor(
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        random_state=2,
    ).fit(x[train_idx], treatment[train_idx])

    outcome_residual[test_idx] = outcome[test_idx] - model_y.predict(x[test_idx])
    treatment_residual[test_idx] = (
        treatment[test_idx] - model_t.predict(x[test_idx])
    )

dml_theta = np.dot(treatment_residual, outcome_residual) / np.dot(
    treatment_residual, treatment_residual
)

# The normalized orthogonal score is an estimated influence function for
# theta under iid sampling in this partially linear model.
moment_residual = treatment_residual * (
    outcome_residual - dml_theta * treatment_residual
)
influence = moment_residual / np.mean(treatment_residual ** 2)
standard_error = influence.std(ddof=1) / np.sqrt(n)
confidence_interval = (
    dml_theta - 1.96 * standard_error,
    dml_theta + 1.96 * standard_error,
)

naive_theta = LinearRegression().fit(
    treatment.reshape(-1, 1), outcome
).coef_[0]

print(f"True theta:       {theta_true:.3f}")
print(f"Naive regression: {naive_theta:.3f}")
print(f"Cross-fitted DML: {dml_theta:.3f}")
print(f"Influence SE:     {standard_error:.3f}")
print(
    "Approximate 95% CI:",
    f"[{confidence_interval[0]:.3f}, {confidence_interval[1]:.3f}]",
)
```

</details>

The score must match the estimand and treatment structure. The residual-on-residual score above targets a constant partially linear coefficient, commonly with a continuous treatment. For a binary-treatment ATE, the interactive-regression-model score is the AIPW score from the previous section; for heterogeneous effects, DML can orthogonalize nuisance functions before fitting a final CATE model.

Orthogonality reduces sensitivity to nuisance estimation; it does not remove omitted confounding or positivity failures. Valid uncertainty also requires regularity, sufficiently fast nuisance convergence, a fold structure that respects clusters or time, and an estimand compatible with the score. Repeatedly tuning nuisance models on the final causal estimate can undo the separation that cross-fitting is meant to create.

#### **Causal Forests**

A causal forest estimates

$$
\tau(x)=\mathbb E[Y(1)-Y(0)\mid X=x]
$$

using tree neighborhoods designed for treatment-effect heterogeneity rather than outcome prediction alone. Modern implementations combine:

- **honest splitting**, where one subsample chooses tree structure and another estimates leaf effects;
- local treatment and outcome centering;
- treatment-effect-oriented split criteria;
- subsampling and forest weights;
- asymptotic variance or bootstrap-style uncertainty under stated conditions.

One useful view is that a forest defines adaptive neighborhood weights $\alpha_i(x)$: training observations that frequently share leaves with $x$ receive larger weight. With cross-fitted outcome and propensity nuisances, a local effect can solve

$$
\sum_i
\alpha_i(x)
\{T_i-\widehat e(X_i)\}
\left[
Y_i-\widehat m(X_i)
-\tau(x)\{T_i-\widehat e(X_i)\}
\right]
=0.
$$

This is different from fitting a random forest to observed outcomes and subtracting two predictions. The splits, residualization, leaf support, and honesty are designed around a local causal moment rather than predictive purity alone.

<div class="diagram-scroll">

![Causal forests use honest tree construction and local treatment-effect estimation.](assets/causal-forest-honesty.svg){fig-alt="A causal forest separates structure and estimation samples, grows trees that seek treatment-effect heterogeneity, and aggregates local neighborhoods into CATE estimates."}

</div>

<details>
<summary><strong>Python: build an educational honest causal forest</strong></summary>

```python
import numpy as np
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(93)
n_train = 4_000
n_test = 1_500

def make_data(n):
    x = rng.normal(size=(n, 5))
    treatment = rng.binomial(1, 0.5, size=n)
    cate = 1.0 + 0.9 * np.tanh(x[:, 0]) - 0.6 * (x[:, 1] > 0)
    baseline = x[:, 0] ** 2 + 0.5 * x[:, 2] - 0.4 * x[:, 3] * x[:, 4]
    outcome = baseline + treatment * cate + rng.normal(scale=1.0, size=n)
    return x, treatment, outcome, cate

x_train, t_train, y_train, _ = make_data(n_train)
x_test, _, _, tau_test = make_data(n_test)

# This compact implementation illustrates honesty:
# one subsample discovers heterogeneous leaves, while a disjoint subsample
# estimates treated-control differences inside those leaves.
n_trees = 250
tree_predictions = np.empty((n_test, n_trees))

for tree_id in range(n_trees):
    subsample = rng.choice(n_train, size=int(0.75 * n_train), replace=False)
    rng.shuffle(subsample)
    midpoint = len(subsample) // 2
    structure_idx = subsample[:midpoint]
    estimation_idx = subsample[midpoint:]

    # With randomized propensity 0.5, this transformed outcome has
    # conditional expectation E[Z|X=x] = tau(x).
    transformed_outcome = (
        2 * (2 * t_train[structure_idx] - 1) * y_train[structure_idx]
    )
    tree = DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=45,
        random_state=tree_id,
    ).fit(x_train[structure_idx], transformed_outcome)

    estimation_leaves = tree.apply(x_train[estimation_idx])
    test_leaves = tree.apply(x_test)
    global_effect = (
        y_train[estimation_idx][t_train[estimation_idx] == 1].mean()
        - y_train[estimation_idx][t_train[estimation_idx] == 0].mean()
    )

    leaf_effect = {}
    for leaf in np.unique(estimation_leaves):
        in_leaf = estimation_leaves == leaf
        leaf_t = t_train[estimation_idx][in_leaf]
        leaf_y = y_train[estimation_idx][in_leaf]
        if np.sum(leaf_t == 1) >= 10 and np.sum(leaf_t == 0) >= 10:
            leaf_effect[leaf] = (
                leaf_y[leaf_t == 1].mean() - leaf_y[leaf_t == 0].mean()
            )
        else:
            leaf_effect[leaf] = global_effect

    tree_predictions[:, tree_id] = np.array(
        [leaf_effect.get(leaf, global_effect) for leaf in test_leaves]
    )

cate_hat = tree_predictions.mean(axis=1)

rmse = np.sqrt(np.mean((cate_hat - tau_test) ** 2))
correlation = np.corrcoef(cate_hat, tau_test)[0, 1]

print(f"True test ATE:       {tau_test.mean():.3f}")
print(f"Estimated test ATE:  {cate_hat.mean():.3f}")
print(f"CATE RMSE:           {rmse:.3f}")
print(f"CATE rank correlation: {correlation:.3f}")
```

</details>

The compact implementation demonstrates sample splitting and heterogeneous leaves; it is not a replacement for production generalized-random-forest or orthogonal causal-forest software, which adds nuisance residualization, specialized splitting, valid variance estimation, and stronger edge-case handling. A causal forest can discover noise heterogeneity when leaves are small or when tuning and evaluation reuse the same data. Report overlap, calibration, subgroup stability, confidence intervals, policy value, and results across honest splits or seeds. Variable importance is not a causal effect of the feature itself.


### **Causal Discovery**

Causal effect estimation begins with an assumed graph or design. **Causal discovery** asks whether aspects of graph structure can be learned from observational or interventional data. The answer is usually an equivalence class, not a uniquely known DAG.

Typical assumptions include:

- the causal Markov property;
- faithfulness, so conditional independencies reflect graph separation;
- causal sufficiency or an explicit latent-variable model;
- independent, identically distributed samples unless time structure is modeled;
- no unmodeled selection bias;
- reliable conditional-independence tests or graph scores;
- acyclicity for DAG-based methods.

Violating these assumptions can reverse or invent edge orientations.

#### **Constraint-Based and Score-Based Methods**

The **PC algorithm** is constraint based:

1. begin with a dense undirected graph;
2. remove edges when conditional-independence tests find a separating set;
3. orient unshielded colliders;
4. propagate directions using orientation rules while avoiding cycles and unsupported colliders.

**Greedy Equivalence Search (GES)** is score based. It searches over Markov-equivalence classes, adding and then deleting edges to improve a penalized likelihood score such as BIC.

When latent common causes may exist, **Fast Causal Inference (FCI)** extends constraint-based reasoning and returns a partial ancestral graph rather than pretending that every common cause was measured. Other methods gain orientation from additional functional assumptions, such as non-Gaussian noise in LiNGAM or additive-noise asymmetry; differentiable methods such as NOTEARS replace combinatorial search with a continuous acyclicity constraint. These assumptions add information, but they also narrow the class of data-generating processes for which the result is valid.

| Method family | Main signal | Typical output | Important assumptions or weakness |
|---|---|---|---|
| PC | Conditional-independence tests | CPDAG | Usually causal sufficiency, faithfulness, reliable tests |
| FCI | Conditional independencies with possible latent causes | PAG | Faithfulness; expensive and unstable with many conditioning sets |
| GES | Penalized likelihood or another decomposable score | CPDAG | Score consistency and search quality |
| LiNGAM / additive-noise methods | Distributional or functional asymmetry | More fully oriented DAG | Correct functional/noise family |
| NOTEARS-style optimization | Smooth fit plus acyclicity constraint | Weighted DAG | Optimization, model-class, thresholding, and acyclicity assumptions |
| Time-series discovery | Lagged dependence and temporal order | Lagged or contemporaneous graph | Stationarity, sampling frequency, autocorrelation model |

A **CPDAG** represents a Markov-equivalence class of DAGs: directed edges have the same orientation in every DAG in the class, while undirected edges are reversible from the observed conditional independencies alone. DAGs are Markov equivalent when they have the same skeleton and the same unshielded colliders. A **PAG** represents an equivalence class when latent confounding or selection may be present; circle, tail, and arrowhead endpoints encode which ancestral relations are unresolved or invariant.

<div class="diagram-scroll">

![Constraint-based and score-based causal discovery return graphs or equivalence classes under assumptions.](assets/causal-discovery-equivalence.svg){fig-alt="Constraint-based discovery removes edges using conditional independence, score-based discovery optimizes graph fit and complexity, and both often identify an equivalence class rather than one DAG."}

</div>

Three-node chains and forks can be Markov equivalent:

$$
X\rightarrow M\rightarrow Y,\qquad
X\leftarrow M\rightarrow Y,\qquad
X\leftarrow M\leftarrow Y
$$

all imply $X\perp Y\mid M$. In contrast, the collider $X\rightarrow M\leftarrow Y$ implies marginal independence but conditional dependence after adjusting for $M$.

<details>
<summary><strong>Python: inspect the conditional-independence signatures of forks, chains, and colliders</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(104)
n = 50_000

def correlation(a, b):
    return np.corrcoef(a, b)[0, 1]

def partial_correlation(a, b, conditioning):
    z = np.asarray(conditioning).reshape(n, -1)
    a_residual = a - LinearRegression().fit(z, a).predict(z)
    b_residual = b - LinearRegression().fit(z, b).predict(z)
    return correlation(a_residual, b_residual)

# Fork: C -> T and C -> Y
c = rng.normal(size=n)
t_fork = 0.9 * c + rng.normal(size=n)
y_fork = -0.8 * c + rng.normal(size=n)

# Chain: X -> M -> Y
x_chain = rng.normal(size=n)
m_chain = 0.9 * x_chain + rng.normal(size=n)
y_chain = 0.8 * m_chain + rng.normal(size=n)

# Collider: X -> S <- Y
x_collider = rng.normal(size=n)
y_collider = rng.normal(size=n)
s_collider = x_collider + y_collider + rng.normal(scale=0.5, size=n)

print(
    "Fork corr(T,Y), partial corr(T,Y|C):",
    f"{correlation(t_fork, y_fork):.3f}",
    f"{partial_correlation(t_fork, y_fork, c):.3f}",
)
print(
    "Chain corr(X,Y), partial corr(X,Y|M):",
    f"{correlation(x_chain, y_chain):.3f}",
    f"{partial_correlation(x_chain, y_chain, m_chain):.3f}",
)
print(
    "Collider corr(X,Y), partial corr(X,Y|S):",
    f"{correlation(x_collider, y_collider):.3f}",
    f"{partial_correlation(x_collider, y_collider, s_collider):.3f}",
)
```

</details>

Conditional independence alone cannot orient every edge. Time order, interventions, multiple environments, non-Gaussian assumptions, additive-noise structure, or domain knowledge may add orientation information.

Finite-sample errors can propagate through the whole graph. A single mistaken independence decision can delete a true adjacency, create a false separating set, orient a false collider, and trigger further incorrect orientations. Results should therefore be checked across test levels, bootstrap samples, variable sets, and plausible domain constraints. Stability is useful evidence, but a stable graph under the same misspecified assumptions can still be wrong.

Discovery and effect estimation are separate tasks. An edge $X\rightarrow Y$ does not report its causal magnitude, and a partially oriented class may contain graphs that imply different adjustment sets. A discovered graph should be treated as a hypothesis to combine with temporal knowledge, experiments, and substantive review, not a license to make unqualified causal claims.


### **Assumptions, Sensitivity, and Failure Modes**

Causal assumptions are not an appendix; they define the claim. A practical analysis should state at least:

| Assumption or design condition | What it protects | Typical failure |
|---|---|---|
| Well-defined treatment and consistency | Interpretable potential outcomes | Multiple treatment versions or changing implementation |
| Exchangeability / no unmeasured confounding | Comparability after adjustment | Hidden severity, intent, eligibility, or selection |
| Positivity / overlap | Empirical support for contrasts | Near-deterministic treatment assignment |
| Correct temporal order | Pre-treatment adjustment | Using post-treatment features or outcome leakage |
| No interference | Unit-level potential outcomes | Network, market, epidemic, or shared-resource effects |
| Accurate measurement | Correct adjustment and outcome | Differential measurement error or proxy confounding |
| Correct sampling and transport | Target-population relevance | Trial volunteers differ from deployment population |

<div class="diagram-scroll">

![The DoWhy workflow separates causal modeling, identification, estimation, and refutation.](assets/dowhy-causal-workflow.png){fig-alt="The DoWhy workflow starts with input data and a causal graph, identifies a target estimand, estimates a causal effect, and refutes or stress-tests the estimate." width="92%"}

</div>

*Workflow image from the official [DoWhy repository](https://github.com/py-why/dowhy/blob/main/docs/images/dowhy-schematic.png), distributed under the repository's [MIT license](https://github.com/py-why/dowhy/blob/main/LICENSE). “Refute” here means looking for evidence that the estimate is fragile; passing a refutation test does not prove the causal assumptions.*

Observable diagnostics include covariate balance, propensity overlap, weight tails, effective sample size, residual patterns, pre-trends, placebo outcomes, and subgroup stability. They can reveal failures but cannot prove no hidden confounding.

Different checks answer different questions:

| Tool | What it can reveal | What it cannot establish |
|---|---|---|
| Balance and overlap diagnostics | Whether the implemented design balanced measured covariates and retained support | Whether important confounders were unmeasured |
| Residual and nuisance-model checks | Gross functional misspecification and unstable predictions | Correct causal direction or adjustment set |
| Placebo and negative-control tests | Certain temporal, selection, or hidden-bias contradictions | Absence of every possible hidden-bias mechanism |
| Sample splitting and repeated seeds | Overfitting and subgroup instability | Identification |
| Confidence intervals | Sampling variability under the model and design | Uncertainty about untestable causal assumptions |
| Sensitivity analysis | How strong a specified violation must be to change the conclusion | The true magnitude or form of the violation |

It is useful to separate **statistical uncertainty** from **identification uncertainty**. A conventional 95% confidence interval asks how the estimator would vary over repeated samples if the design, estimand, measurement, and identifying assumptions were correct. It does not assign 95% probability to “no unmeasured confounding.” Sensitivity ranges, bounds, and alternative causal models must communicate that second layer.

Sensitivity analysis asks how conclusions change under explicit violations:

- Rosenbaum-style hidden-bias parameters for matched studies;
- E-values or bias factors on a risk-ratio scale;
- partial-$R^2$ or robustness values for omitted-variable strength;
- negative-control exposures or outcomes;
- placebo interventions and impossible temporal effects;
- bounds under bounded confounding or missingness;
- alternative adjustment sets and measurement assumptions.

<details>
<summary><strong>Python: map treatment-effect bias under a specified hidden-confounding model</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(115)
n = 40_000
theta_true = 1.5
strengths = [0.0, 0.3, 0.6, 0.9]
rows = []

for effect_on_treatment in strengths:
    for effect_on_outcome in strengths:
        x = rng.normal(size=n)
        hidden_u = rng.normal(size=n)
        treatment = (
            0.8 * x
            + effect_on_treatment * hidden_u
            + rng.normal(size=n)
        )
        outcome = (
            theta_true * treatment
            + 1.2 * x
            + effect_on_outcome * hidden_u
            + rng.normal(size=n)
        )

        # The analyst adjusts for X but cannot observe U.
        estimate = LinearRegression().fit(
            np.column_stack([treatment, x]), outcome
        ).coef_[0]
        rows.append(
            {
                "U->T": effect_on_treatment,
                "U->Y": effect_on_outcome,
                "estimated effect": estimate,
                "bias": estimate - theta_true,
            }
        )

result = pd.DataFrame(rows)
bias_table = result.pivot(index="U->T", columns="U->Y", values="bias")
print("Bias after adjusting for observed X")
print(bias_table.round(3).to_string())
```

</details>

This table is not a test for hidden confounding. It answers a conditional question: if an omitted variable affected treatment and outcome with the specified strengths and functional form, how large would the bias be? Sensitivity conclusions are only as meaningful as the violation model.

Common causal-ML failure modes include:

- optimizing prediction error for a causal estimand;
- selecting adjustment variables from feature importance;
- conditioning on mediators or colliders;
- fitting separate treatment models where one group has little support;
- interpreting CATE rankings as known individual effects;
- tuning on the same outcomes used for subgroup discovery;
- reporting narrow model-based intervals while ignoring identification uncertainty;
- transporting an effect without modeling population differences;
- treating causal discovery output as ground truth.


### **When Causal Reasoning Changes the ML Problem**

Causal reasoning is needed when the model output will guide an action and the goal is the **incremental consequence** of that action. It changes the target in applications such as:

- treatment assignment and personalized medicine;
- pricing, promotions, and retention outreach;
- policy evaluation and resource allocation;
- recommendation systems with feedback loops;
- diagnosing which system component should be changed;
- estimating effects under domain or policy shifts.

<div class="diagram-scroll">

![A causal formulation proceeds from the intervention question to design, estimand, identification, and estimation.](assets/causal-formulation-decision.svg){fig-alt="The workflow first defines the intervention question, then evaluates the design, selects an estimand, establishes identification, and only then chooses an estimator."}

</div>

Use the weakest method justified by the design:

| Situation | Primary target | Suitable starting point |
|---|---|---|
| Only future outcomes matter; actions do not change the data process | Predictive risk | Supervised learning with deployment-valid evaluation |
| Randomized binary treatment; average effect is sufficient | ATE / ITT | Difference in means plus precision adjustment |
| Observational treatment with defensible measured confounders | ATE, ATT, or CATE | Outcome regression, weighting, matching, or doubly robust estimation |
| High-dimensional nuisance functions; low-dimensional causal parameter | Structural coefficient or ATE | Cross-fitted DML / orthogonal score |
| Treatment effect likely varies and overlap is adequate | CATE / policy value | Meta-learner, R-learner, causal forest |
| Assignment follows a threshold, instrument, or policy timing | Design-specific local effect | RD, IV, DiD, or synthetic control |
| Graph itself is uncertain | Equivalence class or causal hypotheses | Discovery plus interventions and domain review |

Before accepting a causal estimate, document:

1. the target trial: eligibility, time zero, strategies, follow-up, outcome, and estimand;
2. the causal graph or potential-outcome assumptions;
3. why the effect is identified;
4. which observations provide overlap;
5. how nuisance models and hyperparameters were cross-fitted;
6. uncertainty from sampling and model fitting;
7. sensitivity to hidden confounding, measurement, missingness, and interference;
8. whether the effect transports to the deployment population;
9. how a policy using the estimate will be evaluated safely.

Prediction answers “what is likely under the current system?” Causal inference answers “what would change under a specified intervention?” Causal machine learning is valuable when flexible prediction supports that second question without replacing the assumptions that make it answerable.

Primary and official resources include Hernán and Robins' free book [Causal Inference: What If](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/), Pearl's overview of [the foundations of causal inference](https://onlinelibrary.wiley.com/doi/10.1111/j.1467-9531.2010.01228.x), the open-access book [Elements of Causal Inference](https://mitpress.mit.edu/9780262344296/elements-of-causal-inference/), [Stanford's Machine Learning & Causal Inference course](https://www.gsb.stanford.edu/faculty-research/labs-initiatives/sil/research/methods/ai-machine-learning/short-course), [MIT Causal Inference](https://computing.mit.edu/cross-cutting/common-ground-for-computing-education/common-ground-subjects/c08-causal-inference/), the [DoWhy four-step workflow](https://petergtz.github.io/dowhy/main/getting_started/index.html), the original [Double/Debiased Machine Learning paper](https://academic.oup.com/ectj/article/21/1/C1/5056401), the [meta-learners paper](https://arxiv.org/abs/1706.03461), and the original work on [causal forests](https://arxiv.org/abs/1510.04342) and [generalized random forests](https://arxiv.org/abs/1610.01271).
